# Cognopolis · Урок M3 — планировщик «сделай топор»

Ты управляешь жителем игры **Cognopolis** не мышкой, а **кодом-агентом**. В M1 житель реагировал на состояние («рюкзак полон → домой»), в M2 — берёг hp. Теперь игрок поручает ему то, чего нет среди действий мира: **«сделай топор»**. Между желанием и первым полезным действием — цепочка зависимостей: лесопилка → топорище → топор. Реактивных правил уже мало — нужен **план**.

**Что построим.** Планировщика: он читает **поручение** из Ратуши, разворачивает цель в **дерево зависимостей**, считает, **чего не хватает** (have-vs-need), исполняет план по одному шагу за ход — и **перепланирует после каждого шага**, так что уже сделанное выпадает из плана само.

**Тир агента:** планировщик (майлстоун игры **M3**). **Дальше по курсу:** LLM-агент (M4).

> ⚙️ **Working-first.** Ноутбук рассчитан на прогон `Run all` без правок — нужно лишь задать `BASE_URL` живого мира и свой `COGNOPOLIS_TOKEN` (из Ратуши). Учебная активность — в секции **«Задачи»**. Базовый планировщик уже строит топор с нуля; твоё дело — прогнать его через кирку, кузницу и живое поручение.

**Ссылки** (подставь адрес своего мира вместо `<BASE_URL>`):
- API-доки (Swagger, кнопка **Authorize**): `<BASE_URL>/docs`
- 👀 Смотреть за своим агентом в браузере: `<BASE_URL>/?token=<твой токен>` (read-only)
- Контракт: житель видит мир **только** через API (`cognopolis_client`).
- Раньше этого — пройди **реактивного сборщика (M1)** (там поручения и авто-разгрузка) и **боевой цикл (M2)**.

## 1. Сетап

Ставим официальный клиент игры из git-URL (PyPI пока нет) и задаём адрес мира.

In [ ]:
%pip install -q "cognopolis-client @ git+https://github.com/ITrubnikov/Train_of_Thought-Cognopolis-client.git"

In [ ]:
import os
from cognopolis_client import Client, GameError

# ⬇️ ЖИВОЙ МИР COGNOPOLIS (публичный инстанс). Можно переопределить переменной COGNOPOLIS_URL.
BASE_URL = os.environ.get("COGNOPOLIS_URL", "https://kindomklaster.com")

# ⬇️ ТВОЙ ТОКЕН ДОСТУПА (полный доступ к твоему жителю).
#   1. Открой BASE_URL в браузере и зарегистрируйся (логин + пароль).
#   2. Ратуша → вкладка «аккаунт» → кнопка «копировать» — это твой токен.
#   3. Вставь его в COGNOPOLIS_TOKEN ниже (или задай переменную окружения / секрет Colab/Kaggle).
TOKEN = os.environ.get("COGNOPOLIS_TOKEN", "")  # ← вставь токен в кавычки, если не используешь env
assert TOKEN, f"Вставь токен: зарегистрируйся на {BASE_URL}, скопируй токен из Ратуши и задай COGNOPOLIS_TOKEN."

c = Client(BASE_URL, token=TOKEN)

# Мягкая проверка связи: если мир недоступен — не пугаем трейсбеком, живые ячейки ниже пропустятся.
WORLD_UP = True
try:
    Client(BASE_URL).get_map()  # GET /map не требует токена
except Exception as e:
    WORLD_UP = False
    print(f"⚠️  Мир {BASE_URL} недоступен ({type(e).__name__}). Живые ячейки пропущу — проверь COGNOPOLIS_URL или попробуй позже.")

print("Мир:", BASE_URL, "| на связи:", WORLD_UP)
# Тир M3 — планировщик: LLM не нужен (он появится на M4).

## 2. Разогрев — база, склад и поручение

«Глаза» планировщика шире, чем у сборщика. В `get_character()` нас теперь интересуют не только позиция и рюкзак (`inventory`), но и **склад** (`stored`), и **здания базы** (`buildings` — Ратуша и склад есть у каждого аккаунта всегда, ур. 1). А `get_assignment()` приносит **поручение** — задачу от игрока (эту механику ты знаешь из M1).

Сначала наденем шляпу **игрока** и выпишем жителю поручение — то же самое делает вкладка «поручения» в Ратуше. Типов поручения пять: `gather` / `defeat` / `build` / `craft` / `errand`. Казалось бы, бери `craft` — но типизированное `craft` называет **один рецепт и количество** (`{"type": "craft", "resource": "axe", "target": 1}`), а «сделай топор **с нуля**» — это не один крафт, а цепочка: построить лесопилку → скрафтить топорище → скрафтить топор. Ни один тип такую цель не выражает.

Поэтому конвенция урока: typed-часть — **первый большой узел будущего плана** (`build sawmill`), а настоящий интент — строкой в `flavor`. Разворачивать цель в дерево — работа агента, а не поля в базе.

In [ ]:
if WORLD_UP:
    print("👀 Смотри за жителем в браузере:", f"{BASE_URL}/?token={TOKEN}")

    ch = c.get_character()
    print("позиция:", (ch["x"], ch["y"]), "| рюкзак:", ch["inventory"], "| склад:", ch["stored"])
    print("здания базы:", {b["kind"]: b["level"] for b in ch["buildings"]})

    # Шляпа ИГРОКА: выписываем поручение (в браузере — Ратуша → вкладка «поручения»).
    a = c.get_assignment()["assignment"]
    if not a or "топор" not in (a.get("flavor") or "").lower():
        a = c.set_assignment(ch["id"], {
            "type": "build", "structure": "sawmill",  # typed-часть: первый большой узел плана
            "flavor": "сделай топор",                 # настоящий интент — свободным текстом
        })["assignment"]
    print("поручение:", a["type"], a.get("structure"), "| flavor:", a.get("flavor"), "| от:", a["assigned_by"])

Дерево зависимостей топора — числами мира:

```
топор (axe)
├─ топорище (axe_handle)   ← craft: 3 дерева
├─ 2 камня
└─ станция: лесопилка ур.1 ← build: 10 дерева + 5 камня (гейт: Ратуша ур.1 — уже есть)
```

Три правила экономики M3, без которых план не сойдётся:

1. **Build и craft платят со СКЛАДА.** Добыча падает в рюкзак; тратится — только склад. Перекладывает одно в другое **шаг на дом (0,0)** — авто-разгрузка из M1. Поэтому в плане всегда есть шаг «домой» между добычей и стройкой.
2. **Выход craft тоже идёт на склад** — топорище и топор переживут даже гибель жителя (рюкзак теряется, склад и здания — нет).
3. **Ошибки самоописательные:** `not_at_station` называет нужную станцию, `not_enough_resources` показывает have-vs-need. Планировщик держит те же знания в голове заранее — в виде дерева.

Итого с чистого аккаунта: **13 дерева + 7 камня**. Зато с топором каждый `gather` дерева даёт +1 (буст инструмента) — план окупается.

## 3. Разбор — паттерн планировщика

Реактивный агент решает «что сделать *сейчас*». Планировщик — «какая **последовательность** приведёт к цели», и честно пересматривает её каждый ход:

```
observe (character + assignment) → plan (дерево → список шагов) → act (ПЕРВЫЙ шаг) → wait → снова observe
```

- **observe** — `get_character()` (рюкзак, склад, здания) + `get_assignment()` (цель от игрока);
- **plan** — чистая функция `make_plan(состояние, цель) → шаги`: в план попадает только **недостающее** (have-vs-need) — готовое топорище на складе и построенная лесопилка выпадают сами;
- **act** — исполняем **только первый шаг** плана: `move` / `gather` / `build` / `upgrade` / `craft`;
- **wait** — `wait_cooldown()`, и цикл повторяется: план пересчитывается заново.

### Такт разный, и ждать надо честно

Раньше в этом месте стояла оговорка «кулдаун общий для всех действий». Она больше не верна, и это важно для петли:

- **шаг** и обмен с постройками — базовый такт, около секунды;
- **добыча** и **бой** — заметно длиннее (сейчас ×10 базы);
- **крафт** — это работа, которая **длится**, и её длительность сервер объявляет **отдельно у каждого рецепта**. Число не заучивай: `get_recipes()` отдаёт у рецепта поле **`seconds`** (до старта), а `get_character()["work"]` — пару **`left_s` / `total_s`** (в процессе, `null` = житель свободен). Типовые такты этого жителя — в `action_seconds`, остаток текущего — в `cooldown`.

Отсюда правило петли: **`wait_cooldown()` — опт-ин**. Клиент запоминает `cooldown` из ответа последнего действия, но сам не спит; уберёшь вызов — получишь не «чуть быстрее», а поток отказов `character_on_cooldown` (сервер их не тормозит: одна плотная петля выдаёт тысячи мусорных запросов на один крафт). И наоборот — самодельный `time.sleep(30)` тоже неверен: он зашивает сегодняшнее число мира в свой код. Спи ровно столько, сколько вернул сервер.

Перепланирование после каждого шага — та самая надёжность: ошибка шага, чужое вмешательство или потерянный рюкзак не ломают агента, а просто дают другой план. Цель приходит из поручения: `flavor` ищется по **каталогу известных слов** («топор» → `axe`). Это lookup, а не «понимание» текста — понимание появится у LLM-агента в M4. Подробный разбор и трейсы — в лекции урока.

## 4. Задачи — прогони планировщик через дерево

Ниже — **рабочий каркас**: каталоги мира (шпаргалка урока — про неё в ячейке), `make_plan()` и петля `run()`. Базовая версия уже строит топор с чистого аккаунта: добывает, банкует, строит лесопилку и крафтит. **Три задачи** (строки для раскомментирования — в конце ячейки с `run()`):

1. **Кирка.** После топора запусти `run("pickaxe")` — план станет заметно короче: лесопилка (общий узел двух веток дерева) уже стоит. Кирка бустит добычу камня, как топор — дерева.
2. **Кузница.** `run("forge")` — в плане впервые появится узел нового типа: `upgrade town_hall` (кузница требует Ратушу **ур. 2**, апгрейд стоит 15 дерева + 10 камня). Дерево выросло вглубь, а код планировщика не изменился ни на строчку.
3. **Поручение из Ратуши.** Смени поручение **руками в браузере** (Ратуша → «поручения», flavor: «сделай кирку») и запусти `run()` без аргументов — цель приедет из поручения. Сверься со строкой «Поручение vs Сейчас» на карточке жителя в «Жителях».

> ⏱ **Прогон идёт минуты, а не секунды, и это не зависание.** Добыча и крафт — длинные такты (см. раздел 3), а план на топор с нуля — это ~35 действий, из которых 20 — добыча. Ориентир на сегодняшнем мире: **4–5 минут** на `run()` с чистого аккаунта; задача 1 (кирка) — около **2,5 минут**, задача 2 (кузница) — около **8 минут**, потому что там 45 добыч. Петля печатает, что именно она делает и сколько это займёт по данным сервера, — по этим строкам и следи. Хочешь смотреть глазами — открой карту в соседней вкладке: у жителя видно дугу готовности и подпись работы.

Ноутбук исполняется и до, и после правок — улучшай постепенно.

In [ ]:
HOME = (0, 0)                                      # дом: шаг сюда авто-банкает рюкзак на склад
RESOURCE_NODE = {"wood": "tree", "stone": "rock"}  # сырьё -> клетка-нода на карте

# Каталоги мира — шпаргалка урока, и она тут НАМЕРЕННО: планировщику дерево нужно ДО первого
# действия, целиком, чтобы посчитать have-vs-need. Машинный GET /recipes существует (c.get_recipes(),
# токен не нужен) — ниже мы берём из него длительности рецептов; discovery «агент сам открывает,
# что умеет» — это тир M4.
RECIPES = {
    "axe_handle": {"inputs": {"wood": 3},                   "station": "sawmill"},
    "axe":        {"inputs": {"axe_handle": 1, "stone": 2}, "station": "sawmill"},
    "pickaxe":    {"inputs": {"axe_handle": 1, "stone": 4}, "station": "sawmill"},
}
BUILD_COST = {"sawmill": {"wood": 10, "stone": 5}, "forge": {"wood": 8, "stone": 12}}
TOWN_HALL_REQ = {"sawmill": 1, "forge": 2}       # tech-tree: какой уровень Ратуши нужен зданию
TOWN_HALL_UPGRADE = {"wood": 15, "stone": 10}    # стоимость апгрейда Ратуши ур.1 -> ур.2

RU = {"axe": "топор", "axe_handle": "топорище", "pickaxe": "кирка", "sawmill": "лесопилка",
      "forge": "кузница", "town_hall": "Ратуша", "wood": "дерево", "stone": "камень"}
GOAL_WORDS = [("кирк", "pickaxe"), ("топорище", "axe_handle"), ("топор", "axe"), ("кузниц", "forge")]

def parse_goal(assignment):
    """flavor поручения -> известная цель. Поиск слова по каталогу (не «понимание» — оно в M4)."""
    text = ((assignment or {}).get("flavor") or "").lower()
    for word, goal in GOAL_WORDS:
        if word in text:
            return goal
    return None

def have(ch, item):
    """Сколько предмета есть ВСЕГО: рюкзак + склад."""
    return ch["inventory"].get(item, 0) + ch["stored"].get(item, 0)

def levels(ch):
    """Уровни зданий базы. Ратуша и склад есть всегда (ур. 1 по умолчанию)."""
    return {b["kind"]: b["level"] for b in ch["buildings"]}

def carried_total(ch):
    return sum(ch["inventory"].values())

def at_home(ch):
    """Житель дома? Одно поле вместо сверки координат: location — "village" | "field" | "mine"."""
    return ch.get("location", "village" if (ch["x"], ch["y"]) == HOME else "field") == "village"

def nearest(ch, tiles, content):
    """Ближайшая клетка с заданным содержимым (по манхэттенскому расстоянию)."""
    nodes = [t for t in tiles if t["content"] == content]
    return min(nodes, key=lambda t: abs(t["x"] - ch["x"]) + abs(t["y"] - ch["y"]))

def on_tile(ch, tx, ty):
    """Житель СТОИТ на этой клетке. Добывать «вплотную» нельзя: gather работает только с
    ресурсной ноды под ногами, иначе сервер отвечает no_resource_here."""
    return (ch["x"], ch["y"]) == (tx, ty)

def step_toward(ch, tx, ty):
    """Один шаг к цели: сперва по X, потом по Y (карта без стен)."""
    if ch["x"] != tx:
        return ch["x"] + (1 if tx > ch["x"] else -1), ch["y"]
    return ch["x"], ch["y"] + (1 if ty > ch["y"] else -1)

In [ ]:
def make_plan(ch, goal):
    """Чистая функция: (состояние, цель) -> список шагов. В плане только НЕДОСТАЮЩЕЕ (have-vs-need)."""
    lvl = levels(ch)
    need = {"wood": 0, "stone": 0}   # суммарное сырьё под все шаги плана
    builds, crafts = [], []
    planned = set()                  # чтобы общий узел (лесопилка) не попал в план дважды

    def ensure_building(kind):
        if lvl.get(kind, 0) >= 1 or kind in planned:
            return                                    # узел уже закрыт — выпадает из плана
        planned.add(kind)
        if lvl["town_hall"] < TOWN_HALL_REQ[kind] and "town_hall" not in planned:
            planned.add("town_hall")                  # tech-tree: сперва Ратуша нужного уровня
            for res, n in TOWN_HALL_UPGRADE.items():
                need[res] += n
            builds.append(("upgrade", "town_hall"))
        for res, n in BUILD_COST[kind].items():
            need[res] += n
        builds.append(("build", kind))

    def ensure_item(item, qty):
        if item in RESOURCE_NODE:                     # лист дерева: сырьё
            need[item] += qty
            return
        missing = qty - have(ch, item)                # готовый компонент выпадает из плана
        if missing <= 0:
            return
        recipe = RECIPES[item]
        for inp, n in recipe["inputs"].items():       # сперва входы...
            ensure_item(inp, n * missing)
        ensure_building(recipe["station"])            # ...потом станция
        crafts.extend([("craft", item)] * missing)

    if goal in RECIPES:
        ensure_item(goal, 1)
    elif goal in BUILD_COST:
        ensure_building(goal)

    gathers = []
    for res in RESOURCE_NODE:                         # have-vs-need: добываем только нехватку
        shortfall = need[res] - have(ch, res)
        if shortfall > 0:
            gathers.append(("gather", res, shortfall))

    steps = []
    if gathers and carried_total(ch) >= ch["inventory_cap"] and not at_home(ch):
        steps.append(("home",))                       # рюкзак полон — сперва разгрузиться
    steps += gathers
    if (builds or crafts) and (gathers or (carried_total(ch) and not at_home(ch))):
        steps.append(("home",))                       # build/craft платят со склада -> занести добычу
    return steps + builds + crafts

def pretty(step):
    if step[0] == "gather":
        return f"добыть {step[2]}×{RU[step[1]]}"
    if step[0] == "home":
        return "домой (авто-банк)"
    verb = {"build": "построить", "upgrade": "улучшить", "craft": "скрафтить"}[step[0]]
    return f"{verb}: {RU[step[1]]}"

In [ ]:
import time

def run(goal=None, rounds=150):
    """observe -> plan -> act (ПЕРВЫЙ шаг) -> wait. Перепланируем после каждого шага."""
    world = c.get_map()                               # ноды сырья статичны — карту берём один раз
    secs = {r["recipe"]: r.get("seconds")             # длительность крафта — из мира: сервер
            for r in c.get_recipes()["recipes"]}      # объявляет её у КАЖДОГО рецепта отдельно
    if goal is None:                                  # цель — из поручения (flavor -> каталог слов)
        goal = parse_goal(c.get_assignment()["assignment"]) or "axe"
    print(f"Цель: {RU.get(goal, goal)} ({goal})")
    last_sig = None
    for _ in range(rounds):
        ch = c.get_character()                        # observe
        plan = make_plan(ch, goal)                    # plan: всегда свежий, из состояния
        if not plan:
            print(f"✅ План пуст — «{RU.get(goal, goal)}» готов. Повторный run() скажет это сразу.")
            return
        sig = [s[:2] for s in plan]
        if sig != last_sig:                           # печатаем план, когда он «схлопнулся»
            print("📋 план:", " → ".join(pretty(s) for s in plan))
            last_sig = sig
        step = plan[0]                                # act: только ПЕРВЫЙ шаг
        reason = f"{pretty(step)} — план из {len(plan)} шагов, цель: {RU.get(goal, goal)}"
        try:
            if step[0] == "gather":
                node = nearest(ch, world["tiles"], RESOURCE_NODE[step[1]])
                if on_tile(ch, node["x"], node["y"]):
                    c.gather(step[1], reason=reason)  # СТОЯ на ноде — добыча (ресурс называем явно)
                else:
                    c.move(*step_toward(ch, node["x"], node["y"]), reason=reason)
            elif step[0] == "home":
                res = c.move(*step_toward(ch, *HOME), reason=reason)
                nch = res["character"]
                banked = res["result"].get("banked")
                if banked:
                    print("  🏠 дом: авто-разгрузка на склад:", banked)
                if at_home(nch) and carried_total(nch) > 0:
                    print("⚠️ Склад полон — рюкзак не поместился целиком. Улучши склад:"
                          " c.upgrade('storehouse') — и перезапусти run().")
                    return
            elif step[0] in ("build", "upgrade"):
                getattr(c, step[0])(step[1], reason=reason)
                print(f"  🏗 {pretty(step)} — готово")
            elif step[0] == "craft":
                eta = secs.get(step[1])               # сколько займёт ЭТОТ рецепт — сервер сказал
                print(f"  🔨 {pretty(step)} — работа на {eta:.0f} с…" if eta else f"  🔨 {pretty(step)}…")
                c.craft(step[1], reason=reason)
                print(f"     готово, {RU[step[1]]} на складе")
        except GameError as e:
            print("  шаг не прошёл:", e.code, "— перепланирую")   # already_built и т.п. — не провал
            now = c.get_character()                   # кулдаун после ОТКАЗА клиенту неизвестен:
            work = now.get("work") or {}              # спрашиваем мир, а не спим наугад
            left = work.get("left_s", now.get("cooldown", 0.0)) or 0.0
            if work:
                print(f"     житель занят: {work.get('action')} {work.get('key') or ''}"
                      f" — осталось {left:.0f} из {work.get('total_s', 0):.0f} с")
            time.sleep(max(left, 1.0))        # пол в секунду: отказ такт не взводит — не лупим сервер
        c.wait_cooldown()                             # wait: спим ровно тот кулдаун, что вернул сервер
    print("⚠️ Раунды кончились. Перезапусти run() — план продолжится с места остановки.")

if WORLD_UP:
    run()   # цель придёт из поручения: flavor «сделай топор» -> axe

# Задача 1 — кирка (план короче: общий узел «лесопилка» уже закрыт). Раскомментируй:
# if WORLD_UP: run("pickaxe")

# Задача 2 — кузница (в плане появится «улучшить: Ратуша» — tech-tree). Раскомментируй:
# if WORLD_UP: run("forge")

# Задача 3 — смени поручение в Ратуше (flavor: «сделай кирку») и дай цели приехать из поручения:
# if WORLD_UP: run()

## 5. Проверка

Критерий приёма: у жителя есть **топор** (рюкзак или склад), профа `carpentry` выросла, а повторный `run()` сразу отвечает «план пуст». Если топора ещё нет — просто перезапусти `run()`: план — функция от состояния, он продолжит с места остановки.

In [ ]:
if WORLD_UP:
    ch = c.get_character()
    axes = have(ch, "axe")
    carpentry_xp = ch["skills"].get("carpentry", {}).get("xp", 0)
    print("топор:", axes, "| здания:", levels(ch), "| carpentry:", ch["skills"].get("carpentry"))
    assert axes >= 1, "Топора пока нет. Перезапусти run() — план продолжится с места остановки (это тоже урок M3)."
    assert carpentry_xp > 0, "Странно: топор есть, а профа carpentry не выросла. Скрафти его сам — перезапусти run()."
    print("✅ Поручение выполнено: топор на складе. Теперь каждый gather дерева даёт +1 — буст инструмента.")
else:
    print("Мир недоступен — проверка пропущена.")

## Наблюдаемость — смотри, как план схлопывается

Открой `BASE_URL/?token=<твой токен>` в соседней вкладке и запусти `run()` ещё раз (например, за киркой):

- **мысль-пузырь** жителя показывает `reason` каждого шага — «домой (авто-банк) — план из 4 шагов, цель: кирка»: видно, как план схлопывается;
- **дуга готовности и поза** под жителем показывают, что он сейчас делает и сколько такту осталось: рубит, тешет камень, работает на станции. Длинный крафт больше не выглядит как зависшая ячейка — работа видна в мире;
- в **«Жителях»** строка **«Поручение vs Сейчас»** сверяет задачу от игрока с реальным занятием агента (и подсветит диагноз, если агент поручение игнорирует);
- в **Хронике** осталась запись «игрок поручил…» — мост «игрок → агент» на глазах;
- на экранах **станций** — «Лесопилка» и «Кузница» — видно, кто у них сейчас работает и что делает; древо технологий (постройки и их гейты по уровню Ратуши) переехало на экран **«Поселение»**.

Те же два факта агент читает кодом, без браузера: `get_character()["location"]` — где житель (`village` / `field` / `mine`), `get_character()["work"]` — чем занят и сколько осталось (`left_s` из `total_s`).

Ручной тык по API — `BASE_URL/docs` (Swagger, кнопка **Authorize**, токен без префикса `Bearer`). Так «1 житель = 1 агент» видно глазами: в одной вкладке агент планирует, в другой — город живёт.